# Polygonize with Geometry Simplification

`polygonize()` converts raster regions into vector polygons. On high-resolution rasters the result can have thousands of vertices per polygon, which slows rendering and inflates file size.

The `simplify_tolerance` parameter simplifies polygon boundaries during polygonization. Two algorithms are available:
- `simplify_method="douglas-peucker"` (default): distance-based, removes vertices that deviate less than the tolerance from the simplified line
- `simplify_method="visvalingam-whyatt"`: area-based, removes vertices that form triangles smaller than the tolerance threshold

Adjacent polygons share identical simplified boundaries, so no gaps or overlaps are introduced.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

from xrspatial import polygonize

## Generate a classified raster

A synthetic land-cover raster with irregular region boundaries.

In [ ]:
rng = np.random.default_rng(42)
shape = (80, 120)

from scipy.ndimage import gaussian_filter
noise = rng.standard_normal(shape)
smooth = gaussian_filter(noise, sigma=8)
classified = np.digitize(smooth, bins=[-0.5, 0.0, 0.5]) + 1

raster = xr.DataArray(classified.astype(np.int32))

fig, ax = plt.subplots(figsize=(10, 6))
raster.plot(ax=ax, cmap="Set2", add_colorbar=True)
ax.set_title("Classified raster (4 land-cover types)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Polygonize without simplification

In [ ]:
col_raw, pp_raw = polygonize(raster)
total_verts_raw = sum(len(r) for rings in pp_raw for r in rings)
print(f"Polygons: {len(pp_raw)}, Total vertices: {total_verts_raw}")

In [ ]:
def plot_polygons(polygon_points, column, title, ax):
    cmap = plt.cm.Set2
    vals = sorted(set(column))
    val_to_color = {v: cmap(i / max(len(vals) - 1, 1)) for i, v in enumerate(vals)}

    patches = []
    colors = []
    for val, rings in zip(column, polygon_points):
        ext = rings[0]
        patches.append(MplPolygon(ext[:, :2], closed=True))
        colors.append(val_to_color[val])

    pc = PatchCollection(patches, facecolors=colors, edgecolors="black",
                         linewidths=0.3)
    ax.add_collection(pc)
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.invert_yaxis()

fig, ax = plt.subplots(figsize=(10, 6))
plot_polygons(pp_raw, col_raw, f"Raw polygons ({total_verts_raw} vertices)", ax)
plt.tight_layout()
plt.show()

## Douglas-Peucker simplification

Increasing tolerance values show the trade-off between fidelity and vertex count. The tolerance is in coordinate units (pixels here).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, tol in zip(axes, [0.5, 1.5, 3.0]):
    col, pp = polygonize(raster, simplify_tolerance=tol)
    n_verts = sum(len(r) for rings in pp for r in rings)
    reduction = 100 * (1 - n_verts / total_verts_raw)
    plot_polygons(pp, col,
                  f"tolerance={tol}  ({n_verts} verts, {reduction:.0f}% reduction)",
                  ax)

plt.suptitle("Douglas-Peucker simplification", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Visvalingam-Whyatt simplification

Area-based simplification removes vertices that contribute the least area change. The tolerance is a minimum triangle area threshold.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, tol in zip(axes, [0.25, 1.0, 3.0]):
    col, pp = polygonize(raster, simplify_tolerance=tol,
                         simplify_method="visvalingam-whyatt")
    n_verts = sum(len(r) for rings in pp for r in rings)
    reduction = 100 * (1 - n_verts / total_verts_raw)
    plot_polygons(pp, col,
                  f"tolerance={tol}  ({n_verts} verts, {reduction:.0f}% reduction)",
                  ax)

plt.suptitle("Visvalingam-Whyatt simplification", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## GeoDataFrame output

`simplify_tolerance` works with all return types including GeoDataFrame.

In [ ]:
gdf = polygonize(raster, simplify_tolerance=1.5, return_type="geopandas",
                 column_name="landcover")
print(gdf.head(10))
print(f"\nAll geometries valid: {gdf.geometry.is_valid.all()}")